# 03 · Gold aggregates
Builds account profiles, scores every transaction with a points-based risk score, writes a tiered alert queue and evaluates rule performance against the IBM laundering labels.

In [0]:
# Setup
from pyspark.sql import functions as F

CAT = "workspace.aml_lakehouse"
s = spark.table(f"{CAT}.silver_transactions")

is_ach     = F.col("payment_format") == "ACH"
cross_bank = F.col("from_bank") != F.col("to_bank")
big_amt    = F.col("amount_paid") >= 1000

total_pos = s.filter(F.col("is_laundering") == 1).count()
base_rate = total_pos / s.count()

In [0]:
# Account profile: sending and receiving behaviour per account
sent = (s.groupBy("from_account_id").agg(
            F.count("*").alias("txn_sent"),
            F.countDistinct("to_account_id").alias("distinct_receivers"),
            F.sum("amount_paid").alias("total_paid"),
            F.avg("amount_paid").alias("avg_paid"),
            F.sum("is_laundering").alias("laundering_sent"))
        .withColumnRenamed("from_account_id", "account_id"))

received = (s.groupBy("to_account_id").agg(
            F.count("*").alias("txn_received"),
            F.countDistinct("from_account_id").alias("distinct_senders"),
            F.sum("amount_received").alias("total_received"),
            F.avg("amount_received").alias("avg_received"),
            F.sum("is_laundering").alias("laundering_received"))
        .withColumnRenamed("to_account_id", "account_id"))

num_cols = ["txn_sent", "distinct_receivers", "total_paid", "avg_paid", "laundering_sent",
            "txn_received", "distinct_senders", "total_received", "avg_received", "laundering_received"]

profile = (sent.join(received, "account_id", "full_outer")
           .fillna(0, subset=num_cols)
           .withColumn("is_laundering_account",
                       ((F.col("laundering_sent") + F.col("laundering_received")) > 0).cast("int")))

profile.write.format("delta").mode("overwrite").saveAsTable(f"{CAT}.gold_account_profile")

In [0]:
# Risk score: points weighted by the lift each signal showed in exploration
profile = spark.table(f"{CAT}.gold_account_profile")
amt = F.col("amount_paid")

scored = (s.join(profile.select(F.col("account_id").alias("from_account_id"),
                                F.col("distinct_receivers").alias("sender_distinct_receivers")),
                 "from_account_id", "left")
          .withColumn("risk_score",
                F.when(is_ach, 3).otherwise(0)
              + F.when(cross_bank, 1).otherwise(0)
              + F.when((amt >= 10000) & (amt < 100000), 3)
                 .when((amt >= 1000) & (amt < 10000), 2)
                 .when(amt >= 100000, 1).otherwise(0)
              + F.when(F.col("sender_distinct_receivers") >= 10, 1).otherwise(0)))

scored.write.format("delta").mode("overwrite").saveAsTable(f"{CAT}.gold_transaction_scores")

In [0]:
# Tiered alert queue (score >= 6)
sc = spark.table(f"{CAT}.gold_transaction_scores")

alerts = (sc.filter(F.col("risk_score") >= 6)
    .withColumn("alert_tier", F.when(F.col("risk_score") == 8, "high")
                               .when(F.col("risk_score") == 7, "medium")
                               .otherwise("low"))
    .withColumn("rule_ach", is_ach.cast("int"))
    .withColumn("rule_cross_bank", cross_bank.cast("int"))
    .withColumn("rule_sender_fanout", (F.col("sender_distinct_receivers") >= 10).cast("int"))
    .select("transaction_id", "txn_ts", "from_account_id", "to_account_id",
            "amount_paid", "payment_currency", "payment_format",
            "risk_score", "alert_tier", "rule_ach", "rule_cross_bank", "rule_sender_fanout",
            "sender_distinct_receivers", "is_laundering"))

alerts.write.format("delta").mode("overwrite").saveAsTable(f"{CAT}.gold_transaction_alerts")

In [0]:
# Rule performance against the laundering labels
def metrics(df, cond, name):
    n = df.filter(cond).count()
    tp = df.filter(cond & (F.col("is_laundering") == 1)).count()
    return (name, n, tp, tp / n if n else 0.0, tp / total_pos, (tp / n) / base_rate if n else 0.0)

rules = [
    ("ACH", is_ach),
    ("ACH + cross-bank", is_ach & cross_bank),
    ("ACH + cross-bank + amount >= 1k", is_ach & cross_bank & big_amt),
    ("Risk score = 8 (high tier)", F.col("risk_score") == 8),
    ("Risk score >= 7 (medium+)", F.col("risk_score") >= 7),
    ("Risk score >= 6 (all alerts)", F.col("risk_score") >= 6),
]
perf = spark.createDataFrame(
    [metrics(sc, cond, name) for name, cond in rules],
    "rule string, flagged long, true_positives long, precision double, recall double, lift double")
perf.write.format("delta").mode("overwrite").saveAsTable(f"{CAT}.gold_rule_performance")
display(perf)